# 🧠 02 - Deep Ensemble Training & Evaluation Walkthrough

This notebook demonstrates the end-to-end training and evaluation pipeline for the heterogeneous 3-CNN ensemble:
1. **Data Augmentation & Batch Generation** (Train, Validation, Test splits)
2. **Base Architecture Instantiation** (EfficientNetB3, ResNet101, DenseNet121)
3. **Model Training Loop & Checkpointing**
4. **Soft-Voting Ensemble Aggregation**
5. **Multi-Class Evaluation & Confusion Matrix**

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to Python path dynamically
NOTEBOOK_DIR = Path(os.getcwd())
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml_engine.architectures.resnet import build_resnet101
from ml_engine.architectures.efficientnet import build_efficientnetb3
from ml_engine.architectures.densenet import build_densenet121
from ml_engine.pipeline.model_trainer import ModelTrainer

print(f"✅ TensorFlow Version: {tf.__version__}")
print(f"✅ GPU Available: {bool(tf.config.list_physical_devices('GPU'))}")

## 1. Configure Directories & Data Generators
Uses ImageDataGenerator for data augmentation on train data and clean rescaling on val/test data.

In [ ]:
BASE_DATA_DIR = REPO_ROOT / "ml_engine" / "data" / "skin_disease_dataset" / "base_dir"
TRAIN_DIR = BASE_DATA_DIR / "train_dir"
VAL_DIR   = BASE_DATA_DIR / "val_dir"
TEST_DIR  = BASE_DATA_DIR / "test_dir"
MODEL_SAVE_DIR = REPO_ROOT / "trained_models"
MODEL_SAVE_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest"
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

print(f"Train Directory Exists: {TRAIN_DIR.exists()}")
print(f"Model Save Directory:   {MODEL_SAVE_DIR}")

## 2. Load Data Generators & Class Weights
Initializes generators from directories and calculates balanced class weights.

In [ ]:
if TRAIN_DIR.exists():
    train_gen = train_datagen.flow_from_directory(
        str(TRAIN_DIR), target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
    )
    val_gen = val_test_datagen.flow_from_directory(
        str(VAL_DIR), target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
    )
    test_gen = val_test_datagen.flow_from_directory(
        str(TEST_DIR), target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
    )

    # Calculate balanced class weights
    classes = np.unique(train_gen.classes)
    weights = compute_class_weight('balanced', classes=classes, y=train_gen.classes)
    class_weights = dict(zip(classes, weights))
    print("\n⚖️ Computed Class Weights:", class_weights)
else:
    print("⚠️ Dataset directories not found. Prepare HAM10000 dataset in ml_engine/data first.")

## 3. Instantiate & Train Base Architectures
Builds ResNet101, EfficientNetB3, and DenseNet121 base models fine-tuned for 7 disease classes.

In [ ]:
# Example instantiation check
trainer = ModelTrainer(patience=5, lr_factor=0.2)

print("Building ResNet101...")
m_resnet = build_resnet101()

print("Building EfficientNetB3...")
m_effnet = build_efficientnetb3()

print("Building DenseNet121...")
m_densenet = build_densenet121()

print("\n✅ All 3 models built successfully with ImageNet transfer learning weights.")

## 4. Soft-Voting Ensemble Evaluation Function
Averages softmax probability vectors across all three neural architectures.

In [ ]:
def evaluate_ensemble(models, generator):
    """Computes soft-voting ensemble probabilities and classification metrics."""
    all_probs = [model.predict(generator) for model in models]
    ensemble_probs = np.mean(all_probs, axis=0)
    y_pred = np.argmax(ensemble_probs, axis=1)
    y_true = generator.classes
    
    target_names = list(generator.class_indices.keys())
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
    plt.title("Ensemble Confusion Matrix", fontsize=14, weight="bold")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

print("✅ Ensemble evaluation function ready.")

## 5. Export to Quantized TFLite for Low-Memory Serving
Converts trained heavy `.h5` model checkpoints into lightweight, quantized `.tflite` flatbuffers using `ml_engine.export.tflite`.
Enforces Dynamic Range Quantization to reduce binary sizes by ~70% and keep runtime memory strictly below 120MB on Render.

In [ ]:
from ml_engine.export.tflite import convert_h5_to_tflite, convert_all_models

# Option A: Convert all trained .h5 models in trained_models directory
converted_files = convert_all_models(models_dir=MODEL_SAVE_DIR, quantize=True)

print("\n🚀 TFLite Models Ready for Production Backend Serving:")
for path in converted_files:
    print(f"   - {path}")